# CroHeal: train the tomato leaf disease model on Colab

Runs the same pipeline as `ml/` locally, on a free Colab GPU:

1. fetch the tomato subsets of **PlantVillage** (training) and **PlantDoc** (field test)
2. two-stage transfer learning (frozen head, then fine-tuning)
3. export **FP32 / FP16 / INT8** LiteRT models
4. evaluate on lab *and* real-world field photos, plus Grad-CAM

*Runtime → Change runtime type → GPU (T4 is enough).*

In [ ]:
REPO = "https://github.com/Nagarjunasagar/CropDisease_Identification_On-Device_MachineLearning.git"  # adjust if renamed
!git clone --depth 1 $REPO croheal
%cd croheal/ml
!pip -q install -r requirements.txt

In [ ]:
!bash scripts/fetch_data.sh

## Train

The default recipe is MobileNetV2 (fast everywhere, quantises well). On a GPU, try
`backbone=efficientnet_b0` or `backbone=mobilenet_v3_large` and more fine-tuning epochs.
The committed `splits/` keep the train/val/test split identical across runs.

In [ ]:
BACKBONE = "mobilenet_v2"   # mobilenet_v2 | mobilenet_v3_large | mobilenet_v3_small | efficientnet_b0
RUN = f"runs/{BACKBONE}"
# stage 1+2 on lab photos, then continue with PlantDoc field photos mixed in
!python -m cropdx.train --set backbone={BACKBONE} batch_size=64 output_dir={RUN}_lab
!python -m cropdx.train --set init_from={RUN}_lab/best.keras head_epochs=0 finetune_epochs=6 finetune_unfreeze_fraction=0.5 field_train_root=data/plantdoc batch_size=64 output_dir={RUN}


## Export to LiteRT and evaluate

In [ ]:
!python -m cropdx.export --model {RUN}/best.keras --out ../models
!python scripts/make_golden.py --models ../models
!python -m cropdx.evaluate --models ../models --legacy ../legacy/2018-model/model_unquant.tflite --report-dir reports
!python -m cropdx.gradcam --model {RUN}/best.keras --out reports/gradcam.png

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open("reports/metrics.md").read()))
for f in ["accuracy.png", "confusion_lab.png", "confusion_field.png", "gradcam.png"]:
    display(Image(f"reports/{f}"))

## Download the models

Copy them into `models/` of your checkout, then run `mobile/tool/sync_models.sh`
for the Flutter app (the Qt edge app reads `models/` directly).

In [ ]:
!cd .. && zip -qr /content/croheal_models.zip models ml/reports
from google.colab import files
files.download("/content/croheal_models.zip")